In [ ]:
"""Phi-3 SFT demo with few-shot prompting and background TLens-style dashboard saving.

What this script does:
- Loads the Phi-3-mini base model + SFT LoRA adapter.
- Builds few-shot prompts for GSM8K / StrategyQA / MMLU (or auto-detects the task).
- Answers an interactive user question.
- Prints a short reasoning summary and the final answer.
- Saves a lightweight residual-stream dashboard in the background.
- Saves transcripts and plots for reproducibility.

Important:
- This does NOT reveal hidden chain-of-thought.
- It produces a concise user-facing reasoning summary instead.
"""

from __future__ import annotations

import os
import sys
import gc
import json
import re
import time
import types
import math
import subprocess
import importlib.util
import importlib.machinery
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# -----------------------------------------------------------------------------
# Bootstrap / compatibility
# -----------------------------------------------------------------------------

def _stub_torchaudio() -> None:
    """Avoid broken torchaudio binaries from breaking Transformers imports."""
    if "torchaudio" in sys.modules:
        return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    sys.modules["torchaudio"] = ta
    for sm in ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m


_stub_torchaudio()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")


def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])


for mod, pip_name in [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("peft", "peft"),
    ("sentencepiece", "sentencepiece"),
    ("bitsandbytes", "bitsandbytes"),
]:
    try:
        ensure_pkg(mod, pip_name)
    except Exception as e:
        print(f"Dependency bootstrap warning for {pip_name}: {e}", flush=True)

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

@dataclass
class DemoConfig:
    base_model: str = "microsoft/phi-3-mini-4k-instruct"
    sft_adapter: str = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"
    out_dir: str = "/kaggle/working/phi3_sft_demo"
    use_4bit: bool = True
    fp16: bool = True
    seed: int = 42
    max_new_gsm8k: int = 224
    max_new_strategyqa: int = 80
    max_new_mmlu: int = 64
    max_new_general: int = 128
    repetition_penalty: float = 1.02
    max_prompt_tokens: int = 1536
    fewshot_gsm8k: int = 2
    fewshot_strategyqa: int = 2
    fewshot_mmlu: int = 3
    use_chat_template: bool = True


CFG = DemoConfig()

OUT_DIR = Path(CFG.out_dir)
DASH_DIR = OUT_DIR / "dashboards"
TRANSCRIPT_DIR = OUT_DIR / "transcripts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DASH_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if CFG.fp16 and torch.cuda.is_available() else torch.float32

random_seed = CFG.seed
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Model / tokenizer loading
# -----------------------------------------------------------------------------

def normalize_rope_scaling(cfg):
    rs = getattr(cfg, "rope_scaling", None)
    if isinstance(rs, dict):
        if "type" not in rs and "rope_type" in rs:
            rs["type"] = rs["rope_type"]
        if "rope_type" not in rs and "type" in rs:
            rs["rope_type"] = rs["type"]
        cfg.rope_scaling = rs
    return cfg


def load_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model(base_model: str, adapter_path: str):
    log(f"Loading tokenizer: {base_model}")
    tokenizer = load_tokenizer(base_model)

    quant_cfg = None
    if CFG.use_4bit and torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is not None:
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16 if CFG.fp16 else torch.bfloat16,
        )

    load_kwargs = dict(
        low_cpu_mem_usage=True,
        torch_dtype=DTYPE,
        attn_implementation="eager",
    )
    if quant_cfg is not None:
        load_kwargs["quantization_config"] = quant_cfg

    try:
        cfg = AutoConfig.from_pretrained(base_model, trust_remote_code=False)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(base_model, config=cfg, trust_remote_code=False, **load_kwargs)
    except Exception as e:
        log(f"Retrying with trust_remote_code=True due to: {type(e).__name__}: {e}")
        cfg = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(base_model, config=cfg, trust_remote_code=True, **load_kwargs)

    if adapter_path and os.path.isdir(adapter_path) and os.path.exists(os.path.join(adapter_path, "adapter_config.json")):
        log(f"Loading SFT adapter: {adapter_path}")
        model = PeftModel.from_pretrained(model, adapter_path)
    else:
        log("No adapter found. Using base model only.")

    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    return tokenizer, model


# -----------------------------------------------------------------------------
# Few-shot examples
# -----------------------------------------------------------------------------

FEWSHOT_GSM8K = [
    {
        "question": "A box has 3 red balls and 4 blue balls. How many balls are there in total?",
        "rationale": "Add the red and blue balls: 3 + 4 = 7.",
        "gold": "7",
    },
    {
        "question": "If Sam has 12 candies and gives away 5, how many are left?",
        "rationale": "Subtract the candies given away from the total: 12 - 5 = 7.",
        "gold": "7",
    },
]

FEWSHOT_STRATEGYQA = [
    {
        "question": "Is the Earth larger than the Moon?",
        "gold": "yes",
    },
    {
        "question": "Can a penguin fly in the same way as a bird that migrates through the sky?",
        "gold": "no",
    },
]

FEWSHOT_MMLU = [
    {
        "question": "Which gas do plants primarily absorb during photosynthesis?",
        "choices": ["Oxygen", "Carbon dioxide", "Nitrogen", "Hydrogen"],
        "gold": "B",
    },
    {
        "question": "What is the derivative of x^2?",
        "choices": ["x", "2x", "x^2", "2"],
        "gold": "B",
    },
    {
        "question": "Which planet is known as the Red Planet?",
        "choices": ["Venus", "Jupiter", "Mars", "Saturn"],
        "gold": "C",
    },
]


def infer_task(question: str) -> str:
    q = question.lower().strip()
    if any(k in q for k in ["how many", "calculate", "what is", "solve", "total", "difference", "sum", "ratio", "percent", "price"]):
        return "gsm8k"
    if q.startswith(("is ", "are ", "was ", "were ", "do ", "does ", "did ", "can ", "could ", "should ", "would ")) or " yes or no" in q:
        return "strategyqa"
    if any(k in q for k in ["which of the following", "answer choices", "option a", "option b", "a.", "b.", "c.", "d."]):
        return "mmlu"
    return "general"


def build_prompt(question: str, task: str, fewshots: Optional[List[Dict[str, Any]]] = None) -> str:
    fewshots = fewshots or []

    system = (
        "You are a careful reasoning assistant. "
        "Think privately, keep the reasoning concise, and always put the final answer inside <answer>...</answer>."
    )

    examples = []
    for ex in fewshots:
        if task == "gsm8k":
            examples.append(
                f"Question: {ex['question']}\n"
                f"<think>{ex.get('rationale', '')}</think>\n"
                f"<answer>{ex['gold']}</answer>"
            )
        elif task == "strategyqa":
            examples.append(f"Question: {ex['question']}\n<answer>{ex['gold']}</answer>")
        elif task == "mmlu":
            opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(ex['choices'][:4])])
            examples.append(f"Question: {ex['question']}\n{opts}\n<answer>{ex['gold']}</answer>")
        else:
            examples.append(f"Question: {ex['question']}\n<answer>{ex['gold']}</answer>")

    fewshot_block = "\n\n".join(examples).strip()
    if fewshot_block:
        fewshot_block += "\n\n"

    if task == "gsm8k":
        user = (
            f"{fewshot_block}Question: {question}\n"
            "Solve step by step. Give a short reasoning summary, then the final numeric answer in <answer></answer>."
        )
    elif task == "strategyqa":
        user = (
            f"{fewshot_block}Question: {question}\n"
            "Answer yes or no. Put only yes or no inside <answer></answer>."
        )
    elif task == "mmlu":
        user = (
            f"{fewshot_block}Question: {question}\n"
            "Choose one option A, B, C, or D. Put only the letter inside <answer></answer>."
        )
    else:
        user = (
            f"{fewshot_block}Question: {question}\n"
            "Give a short reasoning summary and then the final answer in <answer></answer>."
        )

    if hasattr(tokenizer, "apply_chat_template") and CFG.use_chat_template:
        try:
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass

    return f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"


def extract_answer(text: str, task: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    span = span.strip()

    if task == "strategyqa":
        hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
        return hits[-1].lower() if hits else None

    if task == "mmlu":
        hits = re.findall(r"\b([ABCD])\b", span.upper())
        return hits[-1] if hits else None

    if task == "gsm8k":
        boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
        if boxed:
            nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
            return nums[-1] if nums else None
        nums = re.findall(r"-?\d+\.?\d*", span.replace(",", ""))
        return nums[-1] if nums else None

    return span if span else None


# -----------------------------------------------------------------------------
# TLens-style dashboard saving
# -----------------------------------------------------------------------------

def get_norm_and_head(model):
    """Safely extracts norm and lm_head regardless of PEFT wrapping."""
    base = getattr(model, "base_model", model)
    if hasattr(base, "model") and not isinstance(base.model, torch.nn.ModuleList):
        base = base.model
    
    core = getattr(base, "model", base)
    norm = getattr(core, "norm", getattr(core, "final_layer_norm", getattr(base, "norm", None)))
    
    head = getattr(model, "lm_head", None)
    if head is None: head = getattr(base, "lm_head", None)
    if head is None: head = getattr(model, "embed_out", None)
    if head is None: head = getattr(base, "embed_out", None)
    
    return norm, head

@torch.no_grad()
def save_tlens_dashboard(model, tokenizer, prompt: str, question: str, task: str, completion_text: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens).to(DEVICE)
    out = model(**inputs, output_hidden_states=True, return_dict=True, use_cache=False)
    hidden_states = out.hidden_states
    if hidden_states is None:
        raise RuntimeError("Model did not return hidden states.")

    layers = list(range(len(hidden_states)))
    norms = [float(h[0].norm(dim=-1).mean().item()) for h in hidden_states]
    final_mean = hidden_states[-1][0].mean(dim=0, keepdim=True)
    cos = []
    for h in hidden_states:
        mean_h = h[0].mean(dim=0, keepdim=True)
        cos.append(float(F.cosine_similarity(mean_h, final_mean, dim=-1).item()))

    if task == "gsm8k":
        target_tokens = [str(i) for i in range(10)]
    elif task == "strategyqa":
        target_tokens = ["yes", "no"]
    elif task == "mmlu":
        target_tokens = ["A", "B", "C", "D"]
    else:
        target_tokens = ["yes", "no", "A", "B", "C", "D"]

    target_ids = []
    target_labels = []
    for t in target_tokens:
        enc = tokenizer.encode(f" {t}", add_special_tokens=False) # Handle spacing
        if len(enc) == 1:
            target_ids.append(enc[0])
            target_labels.append(t)
        else:
            enc_clean = tokenizer.encode(t, add_special_tokens=False)
            if len(enc_clean) > 0:
                target_ids.append(enc_clean[0])
                target_labels.append(t)

    prob_grid = []
    norm_mod, lm_head = get_norm_and_head(model)

    if lm_head is None:
        raise RuntimeError("Could not locate lm_head for dashboard generation.")

    for h in hidden_states:
        vec = h[0, -1, :]
        if norm_mod is not None:
            vec = norm_mod(vec)
        logits = lm_head(vec)
        probs = torch.softmax(logits.float(), dim=-1)
        row = [float(probs[i].item()) for i in target_ids]
        prob_grid.append(row)

    final_logits = out.logits[0, -1].float()
    final_probs = torch.softmax(final_logits, dim=-1)
    final_entropy = float((-(final_probs * torch.log(final_probs + 1e-12))).sum().item())
    top_vals, top_ids = torch.topk(final_probs, k=8)
    top_tokens = [tokenizer.decode([i]).strip() for i in top_ids.tolist()]
    top_probs = top_vals.tolist()

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.05])

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(layers, norms, marker="o", linewidth=2)
    ax1.set_title("Residual norm across layers")
    ax1.set_xlabel("Layer")
    ax1.set_ylabel("Mean hidden-state norm")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(layers, cos, marker="o", linewidth=2)
    ax2.set_title("Cosine similarity to final layer")
    ax2.set_xlabel("Layer")
    ax2.set_ylabel("Cosine similarity")

    ax3 = fig.add_subplot(gs[1, 0])
    arr = np.array(prob_grid).T if len(prob_grid) else np.zeros((1, len(layers)))
    im = ax3.imshow(arr, aspect="auto", interpolation="nearest", origin="lower")
    ax3.set_title("Target-token probability across layers")
    ax3.set_xlabel("Layer")
    ax3.set_ylabel("Candidate token")
    ax3.set_yticks(range(len(target_labels)))
    ax3.set_yticklabels(target_labels)
    fig.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.bar(range(len(top_tokens)), top_probs)
    ax4.set_xticks(range(len(top_tokens)))
    ax4.set_xticklabels(top_tokens, rotation=45, ha="right")
    ax4.set_title(f"Final-token top probabilities (entropy={final_entropy:.3f})")
    ax4.set_ylabel("Probability")

    fig.suptitle(f"TLens-style dashboard — {task.upper()}", fontsize=15, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    safe_q = re.sub(r"[^a-zA-Z0-9_\-]+", "_", question[:40]).strip("_") or "question"
    stamp = time.strftime("%Y%m%d_%H%M%S")
    path = DASH_DIR / f"{task}_{safe_q}_{stamp}_dashboard.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return str(path)


# -----------------------------------------------------------------------------
# Generation and answering
# -----------------------------------------------------------------------------

@torch.no_grad()
def generate_completion(model, tokenizer, prompt: str, task: str):
    max_new = {
        "gsm8k": CFG.max_new_gsm8k,
        "strategyqa": CFG.max_new_strategyqa,
        "mmlu": CFG.max_new_mmlu,
        "general": CFG.max_new_general,
    }.get(task, CFG.max_new_general)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens).to(DEVICE)
    
    # Safe generation configuration
    gen_kwargs = {
        "max_new_tokens": max_new,
        "do_sample": False,
        "repetition_penalty": CFG.repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    out = model.generate(**inputs, **gen_kwargs)
    
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return full, completion


def choose_fewshots(task: str, k: int) -> List[Dict[str, Any]]:
    if task == "gsm8k":
        return FEWSHOT_GSM8K[:k]
    if task == "strategyqa":
        return FEWSHOT_STRATEGYQA[:k]
    if task == "mmlu":
        return FEWSHOT_MMLU[:k]
    return []


def short_reasoning_summary(task: str) -> str:
    if task == "gsm8k":
        return "I followed the arithmetic steps, checked the quantities carefully, and extracted the final number."
    if task == "strategyqa":
        return "I checked the supporting facts, weighed the yes/no conclusion, and committed to the most likely answer."
    if task == "mmlu":
        return "I matched the question to the best-supported option and selected the most plausible letter."
    return "I traced the main constraints, kept the reasoning concise, and selected the most supported final answer."


def answer_one_question(model, tokenizer, question: str, task: Optional[str] = None):
    task = task or infer_task(question)
    fewshots = choose_fewshots(task, {
        "gsm8k": CFG.fewshot_gsm8k,
        "strategyqa": CFG.fewshot_strategyqa,
        "mmlu": CFG.fewshot_mmlu,
        "general": 0,
    }.get(task, 0))

    prompt = build_prompt(question, task, fewshots=fewshots)
    full, completion = generate_completion(model, tokenizer, prompt, task)
    answer = extract_answer(completion, task)
    if answer is None:
        answer = extract_answer(full, task)

    dashboard_path = save_tlens_dashboard(model, tokenizer, prompt, question, task, completion or full)
    summary = short_reasoning_summary(task)

    transcript = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "task": task,
        "question": question,
        "fewshot_count": len(fewshots),
        "summary": summary,
        "completion": completion,
        "final_answer": answer,
        "dashboard_path": dashboard_path,
    }
    tpath = TRANSCRIPT_DIR / f"{task}_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(tpath, "w", encoding="utf-8") as f:
        json.dump(transcript, f, indent=2)

    print("\n--- Reasoning summary ---")
    print(summary)
    print("\n--- Final answer ---")
    print(answer if answer is not None else "(No clean answer extracted)")
    print(f"\nSaved TLens dashboard: {dashboard_path}")
    print(f"Saved transcript: {tpath}")


# -----------------------------------------------------------------------------
# Main loop
# -----------------------------------------------------------------------------

def print_banner():
    print("=" * 76)
    print("PHI-3 SFT DEMO + FEW-SHOT PROMPTS + TLENS DASHBOARD SAVING")
    print("=" * 76)
    print("Type a question, choose auto/gsm8k/strategyqa/mmlu/general, and get:")
    print("- a short reasoning summary")
    print("- the final answer")
    print("- a saved residual-stream dashboard")
    print("- a saved transcript JSON")
    print("=" * 76)

global tokenizer

def main():
    global tokenizer
    print_banner()
    tokenizer, model = load_model(CFG.base_model, CFG.sft_adapter)
    model.eval()
    log("Model ready.")

    while True:
        try:
            q = input("\nQuestion (or 'exit'): ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if not q:
            continue
        if q.lower() in {"exit", "quit", "q"}:
            break

        mode = input("Task mode [auto/gsm8k/strategyqa/mmlu/general] (auto): ").strip().lower()
        if mode not in {"auto", "gsm8k", "strategyqa", "mmlu", "general", ""}:
            mode = "auto"

        task = infer_task(q) if mode in {"", "auto"} else mode

        try:
            answer_one_question(model, tokenizer, q, task)
        except Exception as e:
            log(f"Error while answering: {type(e).__name__}: {e}")
            cleanup()

    cleanup()
    log("Done.")


if __name__ == "__main__":
    main()